In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

In [2]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
gnb = GaussianNB().fit(X_train, y_train)

print("training done")
for cls, prior in zip(iris.target_names, gnb.class_prior_):
    print(cls, round(prior, 4))

training done
setosa 0.3333
versicolor 0.3333
virginica 0.3333


In [3]:
y_pred = gnb.predict(X_test)
y_prob = gnb.predict_proba(X_test)
acc = accuracy_score(y_test, y_pred)

print()
print("test accuracy", acc * 100)
print(classification_report(y_test, y_pred, target_names=iris.target_names))


test accuracy 96.66666666666667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [4]:
cv_scores = cross_val_score(GaussianNB(), X, y, cv=5)
print("cv scores", (cv_scores * 100).round(2))
print("mean", cv_scores.mean() * 100, "std", cv_scores.std() * 100)

cv scores [ 93.33  96.67  93.33  93.33 100.  ]
mean 95.33333333333334 std 2.666666666666666


In [5]:
def gaussian_pdf(x, mu, var):
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-0.5 * (x - mu) ** 2 / var)

In [6]:
x_sample = X_test[0]
scores = []
for k in range(3):
    likelihood = np.prod([
        gaussian_pdf(x_sample[j], gnb.theta_[k, j], gnb.var_[k, j])
        for j in range(4)
    ])
    scores.append(likelihood * gnb.class_prior_[k])

pred_manual = iris.target_names[np.argmax(scores)]
pred_lib = iris.target_names[gnb.predict([x_sample])[0]]

In [7]:
print("manual calc gives", pred_manual)
print("library gives", pred_lib)
print("same answer?", pred_manual == pred_lib)

manual calc gives setosa
library gives setosa
same answer? True
